# Model Training

In [13]:
# Importing libraries

import os
import json
import pickle
import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
import xgboost as xgb
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

## Loading Feature Sets

In [2]:
# Loading feature sets

X_train = pd.read_parquet('../data/features/X_train.parquet')
y_train = pd.read_parquet('../data/features/y_train.parquet')['isFraud']
X_test = pd.read_parquet('../data/features/X_test.parquet')
y_test = pd.read_parquet('../data/features/y_test.parquet')['isFraud']

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'Train data fraud rate: {y_train.mean().round(4)}')
print(f'Test data fraud rate: {y_test.mean().round(4)}')

X_train: (911804, 109)
X_test: (118108, 109)
Train data fraud rate: 0.5
Test data fraud rate: 0.035


## MLflow Setup

In [4]:
# Setting up MLflow

mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_experiment('fraud-detection')

print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experiment set: fraud-detection')

MLflow tracking URI: http://localhost:5000
Experiment set: fraud-detection


## Training the Model

In [ ]:
# Training the baseline model

os.makedirs('../models', exist_ok=True)

with mlflow.start_run(run_name='xgboost-baseline'):
    mlflow.log_params({'model': 'xgboost-default', 'n_estimators': 100})

    model_baseline = xgb.XGBClassifier(random_state=42, n_jobs=-1)
    model_baseline.fit(X_train, y_train)

    y_pred_proba = model_baseline.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_pred_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_metrics({'auc_roc': auc, 'precision': prec, 'recall': rec, 'f1': f1})
    mlflow.sklearn.log_model(model_baseline, 'model')

    print('=== Baseline Results ===')
    print(f'AUC-ROC:    {auc:.4f}')
    print(f'Precision:  {prec:.4f}')
    print(f'Recall:     {rec:.4f}')
    print(f'F1:         {f1:.4f}')

=== Baseline Results ===
AUC-ROC:    0.9255
Precision:  0.8501
Recall:     0.4612
F1:         0.5980


Baseline metrics are already decent, but the low recall drags the F1 score down. Therefore, the model will be tuned.

In [7]:
# Training the tuned model

params = {
    'n_estimators': 300,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': 1,
    'random_state': 42,
    'n_jobs': -1,
    'eval_metric': 'auc'
}

with mlflow.start_run(run_name='xgboost-tuned'):
    mlflow.log_params(params)

    model_tuned = xgb.XGBClassifier(**params)
    model_tuned.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
        )

    y_pred_proba = model_tuned.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_pred_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_metrics({'auc_roc': auc, 'precision': prec, 'recall': rec, 'f1': f1})
    mlflow.sklearn.log_model(model_tuned, 'model')

    print('=== Tuned Results ===')
    print(f'AUC-ROC:    {auc:.4f}')
    print(f'Precision:  {prec:.4f}')
    print(f'Recall:     {rec:.4f}')
    print(f'F1:         {f1:.4f}')

[0]	validation_0-auc:0.79291
[100]	validation_0-auc:0.87699
[200]	validation_0-auc:0.89421
[299]	validation_0-auc:0.90563
=== Tuned Results ===
AUC-ROC:    0.9056
Precision:  0.7963
Recall:     0.4087
F1:         0.5401


Interestingly enough, the tuned model performs worse than the baseline model. This prompts for adjustments in some parameters, as well as increasing the n_estimators and introducing early stopping to maximise model performance.

In [ ]:
# Training V2 of the tuned model

params_v2 = {
    'n_estimators': 700,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'random_state': 42,
    'n_jobs': -1,
    'early_stopping_rounds': 50,
    'eval_metric': 'auc'
}

with mlflow.start_run(run_name='xgboost-tuned-v2'):
    mlflow.log_params(params_v2)

    model_tuned_v2 = xgb.XGBClassifier(**params_v2)
    model_tuned_v2.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
        )

    y_pred_proba = model_tuned_v2.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_pred_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_metrics({'auc_roc': auc, 'precision': prec, 'recall': rec, 'f1': f1})
    mlflow.sklearn.log_model(model_tuned, 'model')

    print('=== Tuned V2 Results ===')
    print(f'AUC-ROC:    {auc:.4f}')
    print(f'Precision:  {prec:.4f}')
    print(f'Recall:     {rec:.4f}')
    print(f'F1:         {f1:.4f}')

[0]	validation_0-auc:0.79711
[100]	validation_0-auc:0.87692
[200]	validation_0-auc:0.89412
[300]	validation_0-auc:0.90527
[400]	validation_0-auc:0.91381
[500]	validation_0-auc:0.92117
[600]	validation_0-auc:0.92696
[699]	validation_0-auc:0.93214
=== Tuned V2 Results ===
AUC-ROC:    0.9321
Precision:  0.8742
Recall:     0.4689
F1:         0.6104


V2 of the tuned model performs better than the baseline model, but the AUC is still climbing at 699, which means the model has not converged yet and early stopping has still not been triggered. Therefore, the n_estimators parameter will be increased further until early stopping has been triggered.

In [9]:
# Training V3 of the tuned model

params_v3 = {
    'n_estimators': 1500,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'random_state': 42,
    'n_jobs': -1,
    'early_stopping_rounds': 50,
    'eval_metric': 'auc'
}

with mlflow.start_run(run_name='xgboost-tuned-v3'):
    mlflow.log_params(params_v3)

    model_tuned_v3 = xgb.XGBClassifier(**params_v3)
    model_tuned_v3.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
        )

    y_pred_proba = model_tuned_v3.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_pred_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_metrics({'auc_roc': auc, 'precision': prec, 'recall': rec, 'f1': f1})
    mlflow.sklearn.log_model(model_tuned, 'model')

    print('=== Tuned V3 Results ===')
    print(f'AUC-ROC:    {auc:.4f}')
    print(f'Precision:  {prec:.4f}')
    print(f'Recall:     {rec:.4f}')
    print(f'F1:         {f1:.4f}')

[0]	validation_0-auc:0.79711
[100]	validation_0-auc:0.87692
[200]	validation_0-auc:0.89412
[300]	validation_0-auc:0.90527
[400]	validation_0-auc:0.91381
[500]	validation_0-auc:0.92117
[600]	validation_0-auc:0.92696
[700]	validation_0-auc:0.93217
[800]	validation_0-auc:0.93642
[900]	validation_0-auc:0.94016
[1000]	validation_0-auc:0.94358
[1100]	validation_0-auc:0.94679
[1200]	validation_0-auc:0.94927
[1300]	validation_0-auc:0.95128
[1400]	validation_0-auc:0.95317
[1499]	validation_0-auc:0.95477
=== Tuned V3 Results ===
AUC-ROC:    0.9548
Precision:  0.9168
Recall:     0.5388
F1:         0.6788


V3 performs better than V2 and the AUC still keeps climbing. However, marginal increases in AUC without early stopping being triggered will not be ideal, so the learning rate will be slightly increased alongside the n_estimators to make the model converge slightly faster.

In [10]:
# Training V4 of the tuned model

params_v4 = {
    'n_estimators': 2000,
    'max_depth': 6,
    'learning_rate': 0.08,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'random_state': 42,
    'n_jobs': -1,
    'early_stopping_rounds': 50,
    'eval_metric': 'auc'
}

with mlflow.start_run(run_name='xgboost-tuned-v4'):
    mlflow.log_params(params_v4)

    model_tuned_v4 = xgb.XGBClassifier(**params_v4)
    model_tuned_v4.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
        )

    y_pred_proba = model_tuned_v4.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_pred_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_metrics({'auc_roc': auc, 'precision': prec, 'recall': rec, 'f1': f1})
    mlflow.sklearn.log_model(model_tuned, 'model')

    print('=== Tuned V4 Results ===')
    print(f'AUC-ROC:    {auc:.4f}')
    print(f'Precision:  {prec:.4f}')
    print(f'Recall:     {rec:.4f}')
    print(f'F1:         {f1:.4f}')

[0]	validation_0-auc:0.79711
[100]	validation_0-auc:0.88708
[200]	validation_0-auc:0.90598
[300]	validation_0-auc:0.91890
[400]	validation_0-auc:0.92897
[500]	validation_0-auc:0.93622
[600]	validation_0-auc:0.94247
[700]	validation_0-auc:0.94687
[800]	validation_0-auc:0.95047
[900]	validation_0-auc:0.95355
[1000]	validation_0-auc:0.95624
[1100]	validation_0-auc:0.95834
[1200]	validation_0-auc:0.96020
[1300]	validation_0-auc:0.96148
[1400]	validation_0-auc:0.96276
[1500]	validation_0-auc:0.96390
[1600]	validation_0-auc:0.96485
[1700]	validation_0-auc:0.96567
[1800]	validation_0-auc:0.96625
[1900]	validation_0-auc:0.96701
[1999]	validation_0-auc:0.96748
=== Tuned V4 Results ===
AUC-ROC:    0.9675
Precision:  0.9417
Recall:     0.6218
F1:         0.7491


For now, increasing the n_estimators parameter even more would yield diminishing returns, so this model will be declared as the champion model and saved.

## Saving the Champion Model

In [14]:
# Saving the champion model

with open('../models/champion_model.pkl', 'wb') as f:
    pickle.dump(model_tuned_v4, f)

champion_info = {
    'model': 'xgboost-tuned-v4',
    'n_estimators': 2000,
    'learning_rate': 0.08,
    'auc_roc': 0.9675,
    'precision': 0.9417,
    'recall': 0.6218,
    'f1': 0.7491,
    'threshold': 0.5
}

with open('../models/champion_info.json', 'w') as f:
    json.dump(champion_info, f, indent=2)

print('Champion model saved.')
print(f'Location: ../models/champion_model.pkl')

Champion model saved.
Location: ../models/champion_model.pkl


This concludes the model training phase, where the XGBoost model for this transaction fraud detection system has been trained and tuned.